# LLM Outer Loop Pipeline (Kubeflow Pipelines SDK v2)

This notebook defines, compiles, and submits the **outer loop** for production promotion.

## What this pipeline does
- Load a native candidate staged by the inner loop
- Run full validation gate
- Convert to ONNX (serving format)
- Validate ONNX runtime smoke test
- Push deployable artifact to S3
- Register model version in Model Registry
- Deploy from registry
- Run endpoint smoke test

## Prerequisites (pre-created in OpenShift AI)
- Kubeflow Pipelines
- MinIO/S3 bucket + credentials
- Model Registry service
- Serving runtime template (e.g. ONNX/OpenVINO/vLLM)
- Project namespace + RBAC

Everything lives in this notebook — no separate `.py` file is required.

## Pipeline architecture

```text
s3://models/staging/candidates/{run_id}/
        │
        ▼
[1] load_candidate_from_s3 ──> Output[Model]
        │
        ▼
[2] validate_candidate ──> If passed
        │
        ▼
[3] convert_to_onnx ──> Output[Model] (ONNX)
        │
        ▼
[4] validate_onnx_runtime
        │
        ▼
[5] push_deployment_artifact_to_s3
        │   s3://models/artifacts/{model_name}/{version}/
        ▼
[6] register_model_version (Model Registry)
        │
        ▼
[7] deploy_from_registry
        │
        ▼
[8] smoke_test_endpoint
```

**Deployment reads from Model Registry**, not directly from arbitrary S3 paths.

## S3 paths

| Path | Format | Stage |
|------|--------|-------|
| `s3://models/staging/candidates/{run_id}/` | PyTorch/native | Inner loop output (input here) |
| `s3://models/artifacts/{model_name}/{version}/` | ONNX | Outer loop deployable artifact |

## 1) Define pipeline components and orchestration

In [ ]:
from typing import NamedTuple

from kfp import dsl
from kfp.dsl import Input, Model, Output


ValidationResult = NamedTuple(
    "ValidationResult",
    [("passed", bool), ("status_message", str)],
)

RegistryResult = NamedTuple(
    "RegistryResult",
    [
        ("registered_model_name", str),
        ("registered_model_version", str),
        ("artifact_uri", str),
        ("stage", str),
    ],
)

DeploymentResult = NamedTuple(
    "DeploymentResult",
    [("deployment_name", str), ("endpoint_url", str), ("deployed_version", str)],
)

LoadResult = NamedTuple("LoadResult", [("candidate_uri", str)])


@dsl.component(base_image="python:3.11", packages_to_install=["boto3"])
def load_candidate_from_s3(
    bucket_name: str,
    run_id: str,
    s3_endpoint: str,
    model: Output[Model],
) -> LoadResult:
    """Download native candidate artifact from inner-loop staging path."""
    import os

    import boto3
    from botocore.exceptions import BotoCoreError, ClientError

    prefix = f"models/staging/candidates/{run_id.strip('/')}/"
    candidate_uri = f"s3://{bucket_name}/{prefix}"
    os.makedirs(model.path, exist_ok=True)

    s3_client_kwargs = {}
    if s3_endpoint:
        s3_client_kwargs["endpoint_url"] = s3_endpoint

    s3 = boto3.client("s3", **s3_client_kwargs)

    try:
        paginator = s3.get_paginator("list_objects_v2")
        downloaded = 0
        for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
            for item in page.get("Contents", []):
                key = item["Key"]
                if key.endswith("/"):
                    continue
                relative_path = key[len(prefix):]
                local_path = os.path.join(model.path, relative_path)
                os.makedirs(os.path.dirname(local_path), exist_ok=True)
                s3.download_file(bucket_name, key, local_path)
                downloaded += 1

        if downloaded == 0:
            raise FileNotFoundError(f"No objects found at {candidate_uri}")

        print(f"[load-candidate] Downloaded {downloaded} object(s) from {candidate_uri}")
        return LoadResult(candidate_uri)
    except (BotoCoreError, ClientError, FileNotFoundError) as exc:
        # Demo fallback when S3 is unavailable.
        import json

        with open(os.path.join(model.path, "config.json"), "w", encoding="utf-8") as handle:
            json.dump({"source_uri": candidate_uri, "artifact_format": "pytorch"}, handle)
        with open(os.path.join(model.path, "pytorch_model.bin"), "w", encoding="utf-8") as handle:
            handle.write("mock-native-candidate")
        print(f"[load-candidate] S3 unavailable, created mock candidate. Reason: {exc}")
        return LoadResult(candidate_uri)


@dsl.component(base_image="python:3.11")
def validate_candidate(
    model: Input[Model],
    candidate_uri: str,
    min_accuracy: float,
    require_guardrail_pass: bool,
) -> ValidationResult:
    """Full outer-loop validation gate before conversion and promotion."""
    import json
    import os

    config_path = os.path.join(model.path, "config.json")
    if not os.path.exists(config_path):
        return ValidationResult(False, "Candidate missing config.json.")

    with open(config_path, encoding="utf-8") as handle:
        metadata = json.load(handle)

    if require_guardrail_pass and metadata.get("input_guardrail_passed") is False:
        return ValidationResult(False, "Candidate failed input guardrail metadata check.")

    mock_accuracy = float(metadata.get("quick_validation_accuracy", 0.84))
    if mock_accuracy < min_accuracy:
        return ValidationResult(
            False,
            f"Full validation failed: accuracy {mock_accuracy:.2f} < {min_accuracy:.2f}.",
        )

    required_files = ["pytorch_model.bin"]
    missing = [name for name in required_files if not os.path.exists(os.path.join(model.path, name))]
    if missing:
        return ValidationResult(False, f"Missing required candidate files: {missing}")

    return ValidationResult(
        True,
        f"Candidate at {candidate_uri} passed full validation (accuracy={mock_accuracy:.2f}).",
    )


@dsl.component(base_image="python:3.11", packages_to_install=["onnx"])
def convert_to_onnx(
    model: Input[Model],
    onnx_model: Output[Model],
) -> str:
    """Convert native candidate to ONNX serving artifact (mock conversion for demo)."""
    import json
    import os
    import shutil

    os.makedirs(onnx_model.path, exist_ok=True)

    config_path = os.path.join(model.path, "config.json")
    if os.path.exists(config_path):
        shutil.copy(config_path, os.path.join(onnx_model.path, "config.json"))

    # Mock ONNX artifact for demo; replace with real torch.onnx.export in production.
    onnx_path = os.path.join(onnx_model.path, "model.onnx")
    with open(onnx_path, "w", encoding="utf-8") as handle:
        handle.write("mock-onnx-model-bytes")

    metadata = {
        "artifact_format": "onnx",
        "source_model_path": model.path,
        "onnx_file": "model.onnx",
    }
    with open(os.path.join(onnx_model.path, "metadata.json"), "w", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2)

    print(f"[convert-onnx] ONNX artifact created at {onnx_model.path}")
    return onnx_path


@dsl.component(base_image="python:3.11", packages_to_install=["onnxruntime"])
def validate_onnx_runtime(onnx_model: Input[Model]) -> ValidationResult:
    """Smoke test ONNX artifact loads and can run mock inference."""
    import os

    onnx_path = os.path.join(onnx_model.path, "model.onnx")
    if not os.path.exists(onnx_path):
        return ValidationResult(False, "ONNX file model.onnx not found.")

    try:
        import onnxruntime as ort

        # Mock ONNX bytes are not a real graph; treat import success as demo pass.
        _ = ort
        print("[validate-onnx] onnxruntime available; mock ONNX smoke test passed.")
    except Exception as exc:
        return ValidationResult(False, f"ONNX runtime validation failed: {exc}")

    return ValidationResult(True, "ONNX runtime smoke test passed.")


@dsl.component(base_image="python:3.11", packages_to_install=["boto3"])
def push_deployment_artifact_to_s3(
    onnx_model: Input[Model],
    bucket_name: str,
    model_name: str,
    version: str,
    s3_endpoint: str = "",
) -> str:
    """Upload ONNX artifact to deployable S3 path used by registry/deployment."""
    import os

    import boto3
    from botocore.exceptions import BotoCoreError, ClientError

    prefix = f"models/artifacts/{model_name.strip('/')}/{version.strip('/')}/"
    artifact_uri = f"s3://{bucket_name}/{prefix}"

    s3_client_kwargs = {}
    if s3_endpoint:
        s3_client_kwargs["endpoint_url"] = s3_endpoint

    s3 = boto3.client("s3", **s3_client_kwargs)

    uploaded = 0
    for root, _, files in os.walk(onnx_model.path):
        for filename in files:
            local_path = os.path.join(root, filename)
            relative_path = os.path.relpath(local_path, onnx_model.path)
            object_key = f"{prefix}{relative_path}".replace("\\", "/")
            try:
                s3.upload_file(local_path, bucket_name, object_key)
                uploaded += 1
            except (BotoCoreError, ClientError) as exc:
                print(f"[push-artifact] Upload failed; returning target URI for demo. Reason: {exc}")
                return artifact_uri

    print(f"[push-artifact] Uploaded {uploaded} ONNX object(s) to {artifact_uri}")
    return artifact_uri


@dsl.component(base_image="python:3.11")
def register_model_version(
    artifact_uri: str,
    model_name: str,
    version: str,
    registry_url: str,
) -> RegistryResult:
    """Register deployable artifact in Model Registry (mock API for demo)."""
    stage = "Production"
    print(
        f"[model-registry] Registered {model_name}@{version} at {artifact_uri} "
        f"(registry={registry_url}, stage={stage})"
    )
    return RegistryResult(model_name, version, artifact_uri, stage)


@dsl.component(base_image="python:3.11")
def deploy_from_registry(
    registered_model_name: str,
    registered_model_version: str,
    artifact_uri: str,
    namespace: str,
    serving_runtime: str,
) -> DeploymentResult:
    """Deploy registered model version (mock InferenceService creation for demo)."""
    deployment_name = f"{registered_model_name}-{registered_model_version}".replace(".", "-")
    endpoint_url = f"https://{deployment_name}.{namespace}.apps.example.com/v1/models/{registered_model_name}:predict"

    print(
        "[deploy] Creating/updating InferenceService from registry\n"
        f"  name={deployment_name}\n"
        f"  namespace={namespace}\n"
        f"  runtime={serving_runtime}\n"
        f"  artifact_uri={artifact_uri}"
    )
    return DeploymentResult(deployment_name, endpoint_url, registered_model_version)


@dsl.component(base_image="python:3.11", packages_to_install=["requests"])
def smoke_test_endpoint(
    endpoint_url: str,
    test_prompt: str,
) -> ValidationResult:
    """Post-deploy smoke test against serving endpoint (mock for demo)."""
    # In production, call endpoint_url with test_prompt and validate response.
    mock_response = f"Smoke test OK for prompt: {test_prompt}"
    print(f"[smoke-test] Endpoint={endpoint_url} response={mock_response}")
    return ValidationResult(True, "Endpoint smoke test passed.")


@dsl.pipeline(
    name="llm-outer-loop-pipeline",
    description="Outer loop: validate, convert ONNX, push artifact, register, deploy, smoke test.",
)
def llm_outer_loop_pipeline(
    bucket_name: str = "models",
    run_id: str = "run-001",
    model_name: str = "knowledge-base-llm",
    version: str = "v1",
    s3_endpoint: str = "",
    min_accuracy: float = 0.80,
    require_guardrail_pass: bool = True,
    registry_url: str = "https://<model-registry-host>",
    deploy_namespace: str = "kubeflow-user-example-com",
    serving_runtime: str = "onnx-runtime",
    test_prompt: str = "Summarize the product documentation in three bullet points.",
) -> None:
    load_task = load_candidate_from_s3(
        bucket_name=bucket_name,
        run_id=run_id,
        s3_endpoint=s3_endpoint,
    )

    validate_task = validate_candidate(
        model=load_task.outputs["model"],
        candidate_uri=load_task.outputs["candidate_uri"],
        min_accuracy=min_accuracy,
        require_guardrail_pass=require_guardrail_pass,
    )

    with dsl.If(validate_task.outputs["passed"] == True, name="candidate-approved"):
        convert_task = convert_to_onnx(model=load_task.outputs["model"])

        validate_onnx_task = validate_onnx_runtime(onnx_model=convert_task.outputs["onnx_model"])

        with dsl.If(validate_onnx_task.outputs["passed"] == True, name="onnx-approved"):
            push_task = push_deployment_artifact_to_s3(
                onnx_model=convert_task.outputs["onnx_model"],
                bucket_name=bucket_name,
                model_name=model_name,
                version=version,
                s3_endpoint=s3_endpoint,
            )

            register_task = register_model_version(
                artifact_uri=push_task.output,
                model_name=model_name,
                version=version,
                registry_url=registry_url,
            )

            deploy_task = deploy_from_registry(
                registered_model_name=register_task.outputs["registered_model_name"],
                registered_model_version=register_task.outputs["registered_model_version"],
                artifact_uri=register_task.outputs["artifact_uri"],
                namespace=deploy_namespace,
                serving_runtime=serving_runtime,
            )

            smoke_test_endpoint(
                endpoint_url=deploy_task.outputs["endpoint_url"],
                test_prompt=test_prompt,
            )

## 2) Compile pipeline to YAML

In [ ]:
import os

from kfp import compiler

PIPELINE_PACKAGE = "llm_outer_loop_pipeline.yaml"

compiler.Compiler().compile(
    pipeline_func=llm_outer_loop_pipeline,
    package_path=PIPELINE_PACKAGE,
)

print(f"Compiled pipeline to {os.path.abspath(PIPELINE_PACKAGE)}")

## 3) Connect to Kubeflow and submit a run

Use the same `run_id` produced by the inner loop (e.g. `run-001`).

In [ ]:
from kfp.client import Client

KFP_HOST = os.getenv("KFP_HOST", "https://<your-kubeflow-host>/pipeline")
KFP_NAMESPACE = os.getenv("KFP_NAMESPACE", "kubeflow-user-example-com")

client = Client(host=KFP_HOST, namespace=KFP_NAMESPACE)

run_arguments = {
    "bucket_name": "models",
    "run_id": "run-001",
    "model_name": "knowledge-base-llm",
    "version": "v1",
    "s3_endpoint": "https://<minio-route>",
    "min_accuracy": 0.80,
    "require_guardrail_pass": True,
    "registry_url": "https://<model-registry-host>",
    "deploy_namespace": KFP_NAMESPACE,
    "serving_runtime": "onnx-runtime",
    "test_prompt": "Summarize the product documentation in three bullet points.",
}

run = client.create_run_from_pipeline_package(
    pipeline_file=PIPELINE_PACKAGE,
    arguments=run_arguments,
    run_name="llm-outer-loop-demo-run",
    experiment_name="llm-outer-loop",
)

print(f"Submitted run: {run.run_id}")